lendo arquivos dbase.dbf e convertendo para .csv

In [93]:
#%pip install dbfread
import sys
print(sys.executable)


C:\Users\Eduardo\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [94]:
import dbfread
print(f"Versão instalada: {dbfread.__version__}")  


Versão instalada: 2.0.7


In [119]:
#%pip install duckdb
import duckdb



In [124]:
import pandas as pd
from dbfread import DBF    
# instalar pelo terminal a biblioteca : pip install dbfread 
# Summary: Read DBF Files with Python
# Home-page: https://dbfread.readthedocs.io/

import numpy as np 
import os

DbasePath = r'C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados'
DbaseFile = 'PRO00.DBF'

# Concatenando com f-string
# Adicionamos uma barra entre elas se o Path já não terminar com uma
caminho_completo = fr'{DbasePath}\{DbaseFile}'

print("Caminho do arquivo: ",caminho_completo)


Caminho do arquivo:  C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados\PRO00.DBF


In [139]:
import os
#DbasePath = r'C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados'
DbasePath = r'dados'
DbaseFile = 'PRO00.DBF'
caminho_completo2 = os.path.join(DbasePath, DbaseFile)
print(f"Caminho do arquivo os: ({caminho_completo2})  ")


Caminho do arquivo os: (dados\PRO00.DBF)  


In [141]:
import duckdb

# 1. Conecta (ou usa a conexão padrão)
con = duckdb.connect()

# 2. Instala e carrega a extensão necessária para ler DBF
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

# 3. Agora sua consulta vai funcionar
#caminho_completo2 = r'C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados\PRO00.DBF'

# 2. Prepara a busca: remove espaços do codigo_entrada do usuário
codigo_entrada = '587'
codigo_busca_10 = codigo_entrada.strip().rjust(10)   # fica so com os dados, sem espaços e com tamanho de 10 bytes
codigo_busca = codigo_entrada

# Use a conexão 'con' para fazer a query
query = f"SELECT * FROM st_read('{caminho_completo2}') WHERE TRIM(PRO_CODIGO) = '{codigo_busca}'"
#query = f"SELECT * FROM st_read('{caminho_completo2}') WHERE (PRO_CODIGO) = '{codigo_busca}'"   # NAO FUNCIONA
print(query)

registro = con.execute(query).df()

if not registro.empty:
    codigo = registro['PRO_CODIGO'].iloc[0]
    descricao = registro['PRO_DESC'].iloc[0]
    pr_venda = registro['PRO_PVENDA'].iloc[0]
    # Formata primeiro com o ponto e duas casas decimais
    pr_venda_formatado = f"{pr_venda:.2f}"
    # Substitui o ponto pela vírgula para o padrão brasileiro
    pr_venda_final = pr_venda_formatado.replace('.', ',')
    print(f"Código Produto: ({codigo_busca_10}) | Descrição: {descricao} | Preço de Venda: {pr_venda_final}")
    print(f"Código Produto: ({codigo}) | Descrição: {descricao} | Preço de Venda: {pr_venda_final}")
    print(registro)
else:
    print("Produto não encontrado.")


SELECT * FROM st_read('dados\PRO00.DBF') WHERE TRIM(PRO_CODIGO) = '587'
Código Produto: (       587) | Descrição: ESTAÃ├O DE MUSCULAÃ├O EMK 1500 | Preço de Venda: 5240,00
Código Produto: (587) | Descrição: ESTAÃ├O DE MUSCULAÃ├O EMK 1500 | Preço de Venda: 5240,00
   OGC_FID PRO_CODIGO                        PRO_DESC PRO_UNID PRO_CODGRU  \
0        7        587  ESTAÃ├O DE MUSCULAÃ├O EMK 1500       UN       9999   

  PRO_LOCAL  PRO_QUANT  PRO_EMIN  PRO_MARGEM  PRO_PCUSTO  PRO_PVENDA  \
0      None       -2.0         0       81.31      2890.0      5240.0   

   PRO_UCOMP PRO_CODFOR                  PRO_OBS PRO_REFER PRO_CODBAR PRO_AI  \
0 2009-05-09     000013  multiestaþÒo da kenkorp      None       None      A   

   PRO_EMAX  
0         0  


In [127]:
# Comando SQL para buscar o código 104
# Note que usamos rjust(10) dentro do SQL se o dado ainda estiver com espaços
query = f"""
    SELECT PRO_DESC, PRO_PVENDA 
    FROM st_read('{caminho_completo2}') 
    WHERE TRIM(PRO_CODIGO) = '104'
"""

# Executa e transforma em um DataFrame ou lista
resultado = duckdb.query(query).to_df()

if not resultado.empty:
    print(f"Descrição: {resultado['PRO_DESC'].iloc[0]}")
    print(f"Preço: {resultado['PRO_PVENDA'].iloc[0]}")
else:
    print("Produto não encontrado.")


CatalogException: Catalog Error: Table Function with name "st_read" is not in the catalog, but it exists in the spatial extension.

Please try installing and loading the spatial extension:
INSTALL spatial;
LOAD spatial;



In [128]:
try:
    # O parametro ignore_missing_memofile=True ajuda se o .DBF tiver campos Memo (.DBT) faltando
#    tabela = DBF(caminho_completo, ignore_missing_memofile=True)
# Use o 'with' para garantir que o arquivo seja liberado após a leitura
    with DBF(caminho_completo2, ignore_missing_memofile=True) as tabela:
        df = pd.DataFrame(tabela)

    print("Sucesso! O arquivo tem", len(df), "registros.")
    display(df.head()) 
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Sucesso! O arquivo tem 1469 registros.


,PRO_CODIGO,PRO_DESC,PRO_UNID,PRO_CODGRU,PRO_LOCAL,PRO_QUANT,PRO_EMIN,PRO_MARGEM,PRO_PCUSTO,PRO_PVENDA,PRO_UCOMP,PRO_CODFOR,PRO_OBS,PRO_REFER,PRO_CODBAR,PRO_AI,PRO_EMAX
0,102,ANILHA PINTADA DE 02KG VAZADA,UN,1109,,-224.0,20,81.25,6.40,11.6,2018-03-01,000015,NCM=95069100,,,A,100
1,103,ANILHA PINTADA DE 03KG VAZADA,UN,1109,,-161.0,20,81.25,9.60,17.4,2018-03-01,000015,...,,,A,0
2,104,ANILHA PINTADA DE 04KG VAZADA,UN,1109,,-48.0,20,81.25,12.80,23.2,2018-03-01,000015,...,,,A,0
3,587,ESTAÃ├O DE MUSCULAÃ├O EMK 1500,UN,9999,,-2.0,0,81.31,2890.00,5240.0,2009-05-09,000013,multiestaþÒo da kenkorp,,,A,0
4,105,ANILHA PINTADA DE 05KG VAZADA,UN,1109,,-357.0,20,84.83,15.69,29.0,2018-03-01,000015,...,,,A,0


In [97]:
DbasePath = r'C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados'
DbaseFile = 'PRO00.DBF'

# Concatenando com f-string
# Adicionamos uma barra entre elas se o Path já não terminar com uma
caminho_completo = fr'{DbasePath}\{DbaseFile}'
print(DbasePath)
print(DbaseFile)
print(caminho_completo)


C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados
PRO00.DBF
C:\@PROGRAMACAO_01\SENAC-UC2-CASA\dados\PRO00.DBF


In [103]:
print('=============================================')
# Testando a leitura
try:
    # O parametro ignore_missing_memofile=True ajuda se o .DBF tiver campos Memo (.DBT) faltando
#    tabela = DBF(caminho_completo, ignore_missing_memofile=True)
# Use o 'with' para garantir que o arquivo seja liberado após a leitura
    with DBF(caminho_completo, ignore_missing_memofile=True) as tabela:
        df = pd.DataFrame(tabela)

    print("Sucesso! O arquivo tem", len(df), "registros.")
    display(df.head()) 
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Sucesso! O arquivo tem 1469 registros.


,PRO_CODIGO,PRO_DESC,PRO_UNID,PRO_CODGRU,PRO_LOCAL,PRO_QUANT,PRO_EMIN,PRO_MARGEM,PRO_PCUSTO,PRO_PVENDA,PRO_UCOMP,PRO_CODFOR,PRO_OBS,PRO_REFER,PRO_CODBAR,PRO_AI,PRO_EMAX
0,102,ANILHA PINTADA DE 02KG VAZADA,UN,1109,,-224.0,20,81.25,6.40,11.6,2018-03-01,000015,NCM=95069100,,,A,100
1,103,ANILHA PINTADA DE 03KG VAZADA,UN,1109,,-161.0,20,81.25,9.60,17.4,2018-03-01,000015,...,,,A,0
2,104,ANILHA PINTADA DE 04KG VAZADA,UN,1109,,-48.0,20,81.25,12.80,23.2,2018-03-01,000015,...,,,A,0
3,587,ESTAÃ├O DE MUSCULAÃ├O EMK 1500,UN,9999,,-2.0,0,81.31,2890.00,5240.0,2009-05-09,000013,multiestaþÒo da kenkorp,,,A,0
4,105,ANILHA PINTADA DE 05KG VAZADA,UN,1109,,-357.0,20,84.83,15.69,29.0,2018-03-01,000015,...,,,A,0


# 1. Primeiro, vamos ver EXATAMENTE como o dado está no seu DataFrame
exemplo_real = df['PRO_CODIGO'].iloc[0]
print(f"DEBUG - Valor original no arquivo: '{exemplo_real}'")
print(f"DEBUG - Tipo do dado: {type(exemplo_real)}")
x = input("   TECLE ENTER ===> ")   

In [129]:
# Definindo o código que você quer buscar

print(f"\nDigite o codigo do produto para listar. ")
codigo_entrada = input("   Codigo do Produto ===> ")   # inteiro
# 2. Prepara a busca: remove espaços do codigo_entrada do usuário
codigo_busca = codigo_entrada.strip().rjust(10)   # fica so com os dados, sem espaços e com tamanho de 10 bytes
print(f'Buscar codigo:({codigo_busca})')
print(f'               ---------- ')

# Filtrando o DataFrame
# Certifique-se de que o nome da coluna é exatamente PRO_CODIGO
registro = df.loc[df['PRO_CODIGO'] == codigo_busca]    # busca o codigo na coluna PRO_CODIGO e retorna o registro correspondente

if not registro.empty:
    # Acessando o valor da coluna PRO_DESC  PRO_PVENDA .iloc[0]
    codigo = registro['PRO_CODIGO'].iloc[0]
    descricao = registro['PRO_DESC'].iloc[0]
    pr_venda = registro['PRO_PVENDA'].iloc[0]
    # Formata primeiro com o ponto e duas casas decimais
    pr_venda_formatado = f"{pr_venda:.2f}"
    # Substitui o ponto pela vírgula para o padrão brasileiro
    pr_venda_final = pr_venda_formatado.replace('.', ',')
    print(f"Código Produto: ({codigo}) | Descrição: {descricao} | Preço de Venda: {pr_venda_final}")
else:
    print(f"Registro com código {codigo_busca} NÂO encontrado.")


Digite o codigo do produto para listar. 


Buscar codigo:(       106)
               ---------- 
Código Produto: (       106) | Descrição: ANILHA PINTADA DE 10KG VAZADA | Preço de Venda: 55,00


In [100]:
print('=============================================')
url = 'pedidos_narashop.csv'
df = pd.read_csv(url, encoding='utf8', sep=';')
df.head()

,pedido_id,vendedor,categoria,produto,status_pedido,quantidade,valor_unitario,prazo_entrega_dias
0,10001,Rafael Oliveira,Esportes,Raquete de Beach Tennis,Entregue,5,98.01,12
1,10002,Leandro Reis,Eletrônicos,SSD Externo,Entregue,4,599.89,8
2,10003,Flávia Azevedo,Eletrônicos,Notebook,Entregue,5,3774.96,5
3,10004,Lucas Ribeiro,Esportes,Garrafa Térmica,Entregue,1,87.67,4
4,10005,Vinícius Lopes,Eletrônicos,SSD Externo,Entregue,3,439.02,3
